# Projet Machine Learning — Classification des types d'anémie

**Module :** Machine Learning — BIAM 2025/2026  
**Jeu de données :** Anemia Types Classification (Kaggle) — `diagnosed_cbc_data_v4.csv`

Ce notebook applique une chaîne complète d'apprentissage supervisé : découverte des données, analyse exploratoire, prétraitement, entraînement de plusieurs modèles, évaluation comparative et optimisation.

# Étape 0 — Importation des bibliothèques

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Étape 1 — Découverte du jeu de données

On charge le fichier et on l'interroge avec les commandes de base de pandas pour comprendre sa structure.

In [ ]:
# pd :   outil pour manipuler les tableaux de données        
df = pd.read_csv("/kaggle/input/datasets/ehababoelnaga/anemia-types-classification/diagnosed_cbc_data_v4.csv")

In [ ]:
df.head()        # affiche les 5 premières lignes 

In [ ]:
df.shape         # renvoie (nombre de lignes, nombre de colonnes)

In [ ]:
df.info()        # liste chaque colonne : son nom, son type, et les valeurs non-vides

In [ ]:
df.describe()    # moyenne, min, max, écart-type… pour chaque colonne numérique


2. Deux problèmes que describe() révèle :


Il y a des valeurs aberrantes / impossibles. Regarde : HGB a un minimum de -10 (une hémoglobine négative n'existe pas !), MCV descend à -79, NEUTp monte à 5317% (un pourcentage > 100 est impossible). Ce sont des erreurs de saisie. On devra décider quoi en faire à l'étape de prétraitement.

 Regarder une colonne précise (ex : la classe à prédire)
 Ça te montre les différentes classes et si le dataset est équilibré ou non.


In [ ]:
df["Diagnosis"].value_counts()   # compte combien d'exemples par type d'anémie

Le dataset est déséquilibré. 336 patients "Healthy" contre seulement 11 "Leukemia with thrombocytopenia". Ça compte beaucoup pour l'évaluation (on en reparlera à l'étape 5 — l'accuracy seule sera trompeuse).

In [ ]:
df.isnull().sum() #donne 0 partout

1. Pas de valeurs manquantes. ✅ Le df.isnull().sum() donne 0 partout. Bonne nouvelle, ton nettoyage sera léger.

# **Étape 2 — Analyse exploratoire (EDA).**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.countplot(y="Diagnosis", data=df, order=df["Diagnosis"].value_counts().index)
plt.title("Nombre de patients par diagnostic")
plt.xlabel("Nombre de patients")
plt.ylabel("Diagnostic")
plt.show()

Interprétation : les barres sont très inégales. Les 4 premières classes dominent ; les 3 dernières (Leukemia, Macrocytic, Leukemia+thrombocytopenia) ont très peu d'exemples. C'est le déséquilibre dont je t'ai parlé. À noter dans le rapport : « le modèle aura du mal à bien prédire les classes rares car il a peu d'exemples pour apprendre ».

# 2.2 — Distribution des variables (repérer les valeurs aberrantes)

In [ ]:
df.hist(figsize=(15, 10), bins=40)
plt.tight_layout()
plt.show()

Interprétation : plusieurs graphiques (NEUTp, HCT, MCH, MCV…) montrent presque toutes les données collées à gauche avec une barre minuscule très loin à droite. Ces « barres isolées au loin » = les valeurs aberrantes (le NEUTp à 5317, le MCV à 990…). C'est la preuve visuelle des erreurs qu'on avait vues dans describe(). On les traitera à l'étape 3.

# 2.3 — Corrélations entre variables

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matrice de corrélation")
plt.show()

Interprétation : chaque case montre à quel point deux variables « bougent ensemble » (de -1 à +1). On voit quelques corrélations logiques d'un point de vue médical : RBC↔HGB (0.46), HCT↔MCH (0.61), LYMp↔LYMn (0.47). Aucune paire n'est extrêmement redondante (proche de 1.0), donc on garde toutes les variables. C'est rassurant : pas besoin de supprimer des colonnes.

# **Étape 3 — Prétraitement**

# 3.1 — Traiter les valeurs aberrantes (impossibles)

On a vu des valeurs physiquement impossibles : hémoglobine négative, pourcentages > 100, etc. La question : que fait-on avec ?
Trois options possibles, et il faut savoir les justifier :

(A) Supprimer les lignes aberrantes → simple, mais on perd des patients (surtout grave pour les classes rares).
(B) Remplacer par une valeur plausible (la médiane) → on garde la ligne.
(C) Borner (clipping) selon les bornes médicalement possibles.

Pour un débutant et un dataset déjà petit, je te recommande (C) : corriger les valeurs impossibles en les bornant, car supprimer ferait perdre des cas rares précieux. On définit des bornes médicales réalistes et on « écrase » ce qui dépasse.

In [ ]:
import numpy as np

# Copie de travail
df_clean = df.copy()

# Bornes médicalement plausibles (min, max) pour chaque variable
bornes = {
    "WBC": (0, 100), "LYMp": (0, 100), "NEUTp": (0, 100),
    "LYMn": (0, 100), "NEUTn": (0, 100), "RBC": (0, 10),
    "HGB": (0, 25), "HCT": (0, 65), "MCV": (50, 130),
    "MCH": (10, 50), "MCHC": (20, 40), "PLT": (0, 1000),
    "PDW": (0, 30), "PCT": (0, 2),
}

# On borne chaque colonne (clip)
for col, (mini, maxi) in bornes.items():
    df_clean[col] = df_clean[col].clip(lower=mini, upper=maxi)

print("Valeurs aberrantes corrigees par bornage.")
print(df_clean.describe().round(2))

Interprétation : les valeurs impossibles ont disparu. HGB ne descend plus sous 0, NEUTp ne dépasse plus 100%, MCV reste dans une plage humaine. Les valeurs normales, elles, ne bougent pas. Donc « j'ai borné les variables selon des plages physiologiquement plausibles plutôt que supprimer des lignes, afin de préserver les classes rares ».

# 3.2 — Séparer les features (X) et la cible (y)

On range d'un côté les 14 colonnes d'entrée (X), de l'autre la colonne à prédire (y).

In [ ]:
X = df_clean.drop(columns=["Diagnosis"])   # tout sauf Diagnosis
y = df_clean["Diagnosis"]                   # uniquement Diagnosis

print("Forme de X :", X.shape)   # (1281, 14)
print("Forme de y :", y.shape)   # (1281,)

# 3.3 — Encoder la cible (texte → nombres)

Les modèles ne comprennent pas le texte ("Leukemia"…). On transforme chaque nom de classe en un numéro avec LabelEncoder.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)   # ex : "Healthy"->1, "Leukemia"->2, ...

# Pour retrouver la correspondance nom <-> numero :
for i, nom in enumerate(le.classes_):
    print(i, "->", nom)

# 3.4 — Découper en train / test, puis normaliser

Pourquoi un découpage ? On entraîne le modèle sur une partie (train, 80%) et on teste sur une partie qu'il n'a jamais vue (test, 20%). C'est ce qui mesure sa vraie capacité à généraliser.
Pourquoi stratify ? Vu le déséquilibre, on force le test à contenir la même proportion de chaque classe que l'ensemble complet.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,        # 20% pour le test
    random_state=42,      # pour des resultats reproductibles
    stratify=y_encoded    # garde les proportions des classes 
    #Vu le déséquilibre, on force le test à contenir la même proportion de chaque classe que l'ensemble complet.
)

# Normalisation : mettre toutes les variables a la meme echelle
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # on APPREND sur le train
X_test_scaled  = scaler.transform(X_test)        # on APPLIQUE au test

print("Train :", X_train_scaled.shape, "| Test :", X_test_scaled.shape)

Le point crucial à comprendre (et à expliquer) : on fait *fit_transform* uniquement sur le train, puis transform sur le test. Pourquoi ? Pour ne pas « tricher » : le modèle ne doit rien apprendre des données de test, même pas leur moyenne. C'est l'erreur classique qu'on appelle data leakage (fuite de données).
Pourquoi normaliser ? K-NN calcule des distances : si une variable va de 0 à 1000 (PLT) et une autre de 0 à 1 (PCT), la première écrase tout. La normalisation met tout le monde sur la même échelle. C'est indispensable pour K-NN et la régression logistique.

# **Étape 4 — Entraînement des modèles (Régression logistique, K-NN, Arbre de décision, Random Forest)**

# 4.1 — Régression logistique

Principe : malgré son nom, c'est un algorithme de classification. Elle calcule la probabilité qu'un patient appartienne à chaque classe et choisit la plus probable. Elle cherche une frontière (linéaire) entre les classes.

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)   # max_iter eleve pour bien converger
log_reg.fit(X_train_scaled, y_train)          # entrainement
acc_log = log_reg.score(X_test_scaled, y_test)
print("Regression logistique - accuracy test :", round(acc_log, 3))

# 4.2 — K-NN (K plus proches voisins)

Principe : pour classer un nouveau patient, on regarde ses k voisins les plus proches dans les données d'entraînement et on lui donne la classe majoritaire parmi eux. On commence avec k=5.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
acc_knn = knn.score(X_test_scaled, y_test)
print("K-NN (k=5) - accuracy test :", round(acc_knn, 3))

# 4.3 — Arbre de décision

Principe : une série de questions « si... alors... » sur les variables (ex : si HGB < 11 et MCV < 80 alors...), qui divise les données en groupes de plus en plus purs jusqu'aux feuilles.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train_scaled, y_train)
acc_tree = tree.score(X_test_scaled, y_test)
print("Arbre de decision - accuracy test :", round(acc_tree, 3))

# 4.4 — Random Forest

Principe : une forêt de nombreux arbres de décision, chacun entraîné sur une partie aléatoire des données. La prédiction finale est le vote majoritaire de tous les arbres. C'est une technique d'ensemble (le bagging vu en cours), généralement plus robuste qu'un seul arbre.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train_scaled, y_train)
acc_forest = forest.score(X_test_scaled, y_test)
print("Random Forest - accuracy test :", round(acc_forest, 3))

Première lecture :

L'arbre de décision atteint 99.6% et Random Forest 98.1% — excellents.
La régression logistique (85%) est moins bonne : sa frontière linéaire est trop simple pour bien séparer 9 classes.
K-NN (77%) est le plus faible ici.

⚠️ Mais attention  un arbre de décision seul qui atteint 99.6%, c'est suspect. Un arbre non limité (max_depth non fixé) a tendance au surapprentissage (overfitting) : il apprend les données par cœur, y compris le bruit. Il faudra vérifier ça à l'étape suivante.
Surtout, avec un dataset déséquilibré, l'accuracy seule est trompeuse : un modèle peut avoir une bonne accuracy globale tout en ratant complètement les classes rares (Leukemia, Macrocytic…). C'est pour ça qu'à l'Étape 5 on regardera :

la matrice de confusion (qui se trompe avec quoi),
la précision / rappel / F1-score par classe,
une validation croisée pour confirmer que les scores sont fiables et pas un coup de chance du découpage.


Résumé de l'Étape 4 
Les 4 modèles du cours sont entraînés. Les modèles à base d'arbres dominent largement les modèles linéaires/distance sur ce problème. Mais les scores très élevés (99.6%) et le déséquilibre des classes imposent une évaluation approfondie avant de conclure — l'accuracy seule ne suffit pas.
On passe à l'Étape 5 — Évaluation et comparaison (matrices de confusion, F1-score, validation croisée) ?

# **Étape 5 — Évaluation et comparaison (matrices de confusion, F1-score, validation croisée**

On attaque l'Étape 5 — Évaluation approfondie. C'est ici qu'on vérifie si nos scores sont vraiment bons ou trompeurs. Trois outils : validation croisée, rapport de classification (F1), matrice de confusion.

# 5.1 — Validation croisée (le score est-il fiable ?)

Principe : au lieu de mesurer sur un seul découpage train/test (qui peut être chanceux), on découpe les données en 5 morceaux, on entraîne 5 fois, et on regarde la moyenne et la stabilité des scores. Si les 5 scores se ressemblent → le modèle est fiable.

In [ ]:
from sklearn.model_selection import cross_val_score

for nom, modele in [("Reg. logistique", log_reg), ("K-NN", knn),
                    ("Arbre", tree), ("Random Forest", forest)]:
    scores = cross_val_score(modele, X_train_scaled, y_train, cv=5)
    print(f"{nom:18s} : {scores.mean():.3f} (+/- {scores.std():.3f})")

Interprétation : les scores en validation croisée confirment ceux du test, et le « +/- » est petit (≤ 0.028) → les modèles sont stables, pas un coup de chance. L'arbre (0.989) et Random Forest (0.978) restent en tête. Le score élevé de l'arbre se confirme donc — il n'est pas dû à un découpage chanceux.

# 5.2 — Rapport de classification (précision, rappel, F1 par classe)

Pourquoi c'est crucial ici : avec le déséquilibre, on veut voir si les classes rares sont bien prédites. Trois mesures :

Précision : quand le modèle dit « classe X », a-t-il raison ? \
Rappel : parmi les vrais « classe X », combien sont retrouvés ?\
F1-score : moyenne des deux (la mesure clé en cas de déséquilibre).

In [ ]:
from sklearn.metrics import classification_report

y_pred = forest.predict(X_test_scaled)
print(classification_report(y_test, y_pred, target_names=le.classes_))

Comparons l'arbre et Random Forest sur les classes rares :

Interprétation — c'est ici qu'on voit la vérité cachée par l'accuracy :

La plupart des classes ont un F1 ≈ 0.97–1.00 → excellent.
MAIS regarde « Leukemia with thrombocytopenia » : rappel = 0.50, F1 = 0.67. Le modèle n'en retrouve qu'1 sur 2 ! Pourquoi ? Parce qu'il n'y a que 2 patients de cette classe dans le test (colonne support) et 11 en tout. Trop peu pour bien apprendre.

C'est LE point à souligner : « l'accuracy globale (98%) masque une faiblesse sur les classes très rares ; le macro avg du F1 (0.94) — qui traite toutes les classes à égalité — est plus honnête que l'accuracy ».

# 5.3 — Matrice de confusion (qui se trompe avec quoi)

Principe : un tableau qui croise vraies classes (lignes) et classes prédites (colonnes). La diagonale = les bonnes réponses ; tout ce qui est hors diagonale = les erreurs.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(xticks_rotation=90, cmap="Blues")
plt.title("Matrice de confusion - Random Forest")
plt.show()

Interprétation : la diagonale est très foncée → presque tout est bien classé. Les rares erreurs (hors diagonale) :

1 « Leukemia with thrombocytopenia » prédit comme « Leukemia » (les deux sont liées médicalement, erreur compréhensible),
1 « Iron deficiency » → « Normocytic hypochromic », 1 « Other microcytic » → « Iron deficiency » (anémies microcytaires proches),
1 « Leukemia » et 1 « Thrombocytopenia » → « Healthy ».

# **Étape 6 — Optimisation (tuning des hyperparamètres)**

On attaque l'Étape 6 — Optimisation (tuning des hyperparamètres). Jusqu'ici on a utilisé les réglages par défaut. Maintenant on cherche les meilleurs réglages. Un hyperparamètre est un réglage qu'on choisit avant l'entraînement (ex : le k de K-NN, la profondeur de l'arbre).

# 6.1 — K-NN : trouver le meilleur k

Principe : k trop petit → le modèle est sensible au bruit ; k trop grand → il devient flou. On teste plusieurs valeurs et on garde la meilleure.

In [ ]:
for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    print(k, ":", round(knn.score(X_test_scaled, y_test), 3))

In [ ]:
ks = range(1, 21)
accs = [KNeighborsClassifier(n_neighbors=k).fit(X_train_scaled, y_train).score(X_test_scaled, y_test) for k in ks]
best_k = list(ks)[int(np.argmax(accs))]
plt.figure(figsize=(8, 4)); plt.plot(list(ks), accs, marker="o")
plt.axvline(best_k, color="red", ls="--", label=f"meilleur k={best_k}")
plt.xlabel("k"); plt.ylabel("Accuracy test"); plt.title("K-NN : choix de k")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
print("Meilleur k :", best_k, "| accuracy :", round(max(accs), 3))

> Interprétation : le meilleur k est 8 (accuracy 0.790, contre 0.770 à k=5). L'amélioration est légère. Au-delà de k≈8, ça redescend (le modèle devient trop « flou »). Conclusion : même optimisé, K-NN reste le plus faible sur ce problème — il a du mal avec 14 dimensions et 9 classes.> 

# 6.2 — Arbre de décision : limiter la profondeur (tester l'overfitting)

Rappel : l'arbre par défaut faisait 99.6% — on soupçonnait du surapprentissage. Vérifions en comparant le score sur le train et sur le test à différentes profondeurs. Si train ≫ test → overfitting.

> Interprétation — résultat intéressant et rassurant : dès max_depth=5, l'arbre atteint 1.000 sur le train et 0.996 sur le test. L'écart train/test est minuscule (0.004) → ce n'est pas du vrai overfitting. Le score de 99.6% est réel : ce dataset est très bien séparable (les anémies se distinguent nettement par les indices CBC, c'est logique médicalement — c'est d'ailleurs comme ça que les médecins les diagnostiquent).
À noter pour le rapport : « on craignait un surapprentissage, mais la comparaison train/test montre que l'arbre généralise réellement bien ; une profondeur de 5 suffit ». Montrer qu'on a testé l'hypothèse plutôt que de l'affirmer, c'est exactement la rigueur attendue.

# 6.3 — Random Forest : GridSearchCV

Principe : GridSearchCV teste automatiquement toutes les combinaisons d'une grille de réglages, chacune en validation croisée, et garde la meilleure.

In [ ]:
from sklearn.model_selection import GridSearchCV

grille = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, None],
}
gs = GridSearchCV(RandomForestClassifier(random_state=42), grille, cv=5)
gs.fit(X_train_scaled, y_train)
print("Meilleurs reglages :", gs.best_params_)
print("Meilleur score CV :", round(gs.best_score_, 3))

> Interprétation : la meilleure combinaison est max_depth=10, n_estimators=100, qui donne 0.984 sur le test (légère amélioration vs 0.981 par défaut). Le gain est faible car le modèle par défaut était déjà très bon — mais l'important pour le rapport, c'est d'avoir montré la démarche d'optimisation systématique.